In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix

# Reading test, validation, and training data from files

df_test = pd.read_csv('/workspaces/churn-analysis/datasets/test.csv')
df_val = pd.read_csv('/workspaces/churn-analysis/datasets/val.csv')
df_train = pd.read_csv('/workspaces/churn-analysis/datasets/val.csv')

y_test = df_test['churn']
y_val = df_val['churn']
y_train = df_train['churn']

# First rows of the dataframes
print("Test Data:")
print(df_test.head(3))
      

df_test.drop(columns=['churn'], inplace=True)
df_val.drop(columns=['churn'], inplace=True)
df_train.drop(columns=['churn'], inplace=True) 

y_val = (y_val=='yes').astype(int)
y_test = (y_test=='yes').astype(int)



Test Data:
   gender  seniorcitizen partner dependents  tenure phoneservice  \
0  female              0      no         no      41          yes   
1  female              1      no         no      66          yes   
2  female              0      no         no      12          yes   

  multiplelines internetservice onlinesecurity onlinebackup deviceprotection  \
0            no             dsl            yes           no              yes   
1           yes     fiber_optic            yes           no               no   
2            no             dsl             no           no               no   

  techsupport streamingtv streamingmovies        contract paperlessbilling  \
0         yes         yes             yes        one_year              yes   
1          no         yes             yes        two_year              yes   
2          no          no              no  month-to-month              yes   

               paymentmethod  monthlycharges  totalcharges churn  
0  bank_transfe

In [3]:
train_dicts = df_train.to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

test_dicts = df_test.to_dict(orient='records')
X_test = dv.transform(test_dicts)

dv.feature_names_

['contract=month-to-month',
 'contract=one_year',
 'contract=two_year',
 'dependents=no',
 'dependents=yes',
 'deviceprotection=no',
 'deviceprotection=no_internet_service',
 'deviceprotection=yes',
 'gender=female',
 'gender=male',
 'internetservice=dsl',
 'internetservice=fiber_optic',
 'internetservice=no',
 'monthlycharges',
 'multiplelines=no',
 'multiplelines=no_phone_service',
 'multiplelines=yes',
 'onlinebackup=no',
 'onlinebackup=no_internet_service',
 'onlinebackup=yes',
 'onlinesecurity=no',
 'onlinesecurity=no_internet_service',
 'onlinesecurity=yes',
 'paperlessbilling=no',
 'paperlessbilling=yes',
 'partner=no',
 'partner=yes',
 'paymentmethod=bank_transfer_(automatic)',
 'paymentmethod=credit_card_(automatic)',
 'paymentmethod=electronic_check',
 'paymentmethod=mailed_check',
 'phoneservice=no',
 'phoneservice=yes',
 'seniorcitizen',
 'streamingmovies=no',
 'streamingmovies=no_internet_service',
 'streamingmovies=yes',
 'streamingtv=no',
 'streamingtv=no_internet_servic

In [ ]:
for C in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:

    model = LogisticRegression(max_iter=5000, C=C)
    model.fit(X_train, y_train) 

    for k in np.arange(0, 1.0, 0.05):
        y_pred = (model.predict_proba(X_val)[:, 1] >= k).astype(int)
        print(f"C={C}: k={round(k, 2)}, accuracy={round((y_pred == y_val).mean(), 2)}")

Final model accuracy: 0.8
0.8608658876918948


In [11]:
# Final model with C=1 and k=0.4
final_model = LogisticRegression(max_iter=5000, C=1)
final_model.fit(X_train, y_train)
y_pred_final = (final_model.predict_proba(X_val)[:, 1] >= 0.4).astype(int)
print(f"Final model accuracy: {round((y_pred_final == y_val).mean(), 2)}")
print("Classification Report:")
print(classification_report(y_val, y_pred_final))

Final model accuracy: 0.8
Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.85      0.86      1023
           1       0.62      0.67      0.64       386

    accuracy                           0.80      1409
   macro avg       0.75      0.76      0.75      1409
weighted avg       0.80      0.80      0.80      1409

